# Who Can Pay for the Transition

Part A: Canada's energy transition from Statistics Canada. Part B: credit risk models on the 12 scorecard companies. Joint conclusion: which companies can pay for the transition.

Run the cells in order, from the energy-transition folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
sc = pd.read_csv('data/scorecard.csv')
sc.sort_values('total', ascending=False)

In [ ]:
# Part A: headline figures from the StatCan analysis files
el = pd.read_csv('analysis_electricity.csv')
en = pd.read_csv('analysis_energy_production.csv')
cx = pd.read_csv('analysis_capex.csv')
w15 = el[el.year == 2016]['Wind'].iloc[0]  # wind in 2016, the first split year
print('Wind 2016 to 2024, TWh:', round(el[el.year == 2024]['Wind'].iloc[0], 2))
print('Solar 2016 to 2024, TWh:', round(el[el.year == 2024]['Solar'].iloc[0], 2))
print('Fossil share of primary production 2015:', round(en[en.year == 2015]['Fossil share of primary %'].iloc[0], 1), '%')
print('Fossil share of primary production 2024:', round(en[en.year == 2024]['Fossil share of primary %'].iloc[0], 1), '%')
print('Oil and gas capex 2024, $m:', round(cx[cx.year == 2024]['Oil and gas extraction [211]'].iloc[0]))
print('Utilities capex 2024, $m:', round(cx[cx.year == 2024]['Utilities [22]'].iloc[0]))

In [ ]:
# Part B: model results for 2024
mod = pd.read_csv('credit/data/models.csv')
met = pd.read_csv('credit/data/metrics.csv')
m24 = met[met.fiscal_year_end.str.startswith('2024')].set_index('ticker')
mod[['ticker', 'altman_zprime', 'springate', 'zmijewski_prob_pct',
     'merton_dd', 'synthetic_rating', 'debt_ebitda', 'interest_coverage']].round(2)

In [ ]:
# Joint table: scorecard total against debt to EBITDA and coverage
joint = sc.merge(mod[['ticker', 'debt_ebitda', 'interest_coverage',
                      'zmijewski_prob_pct', 'synthetic_rating']],
                 left_on='ticker', right_on='ticker')
joint[['company', 'total', 'debt_ebitda', 'interest_coverage',
       'zmijewski_prob_pct', 'synthetic_rating']].sort_values('total', ascending=False).round(2)

In [ ]:
# Chart: debt to EBITDA, 2024
s = mod.sort_values('debt_ebitda')
plt.figure(figsize=(9, 5.5))
plt.barh(s['company'], s['debt_ebitda'], color='#1D5C30')
plt.axvline(3.0, color='#9CA3AF', linestyle='--')
plt.xlabel('Total debt divided by EBITDA, times')
plt.title('Debt to EBITDA, 2024 annuals')
plt.tight_layout()
plt.show()

In [ ]:
# Chart: leverage against coverage, 2024, log scale
plt.figure(figsize=(9, 5.5))
plt.scatter(mod['debt_ebitda'], mod['interest_coverage'], s=90, color='#1D5C30')
for _, r in mod.iterrows():
    plt.annotate(r['ticker'], (r['debt_ebitda'], r['interest_coverage']),
                 textcoords='offset points', xytext=(5, 5), fontsize=9)
plt.axvline(3.0, color='#9CA3AF', linestyle='--')
plt.axhline(3.0, color='#9CA3AF', linestyle='--')
plt.xlabel('Debt to EBITDA, times')
plt.ylabel('Interest coverage, times (log scale)')
plt.yscale('log')
plt.title('Leverage against interest coverage, 2024 annuals')
plt.tight_layout()
plt.show()

## Method and limits

Scorecard: 4 measures scored 0 to 5 from each company's 2024 sustainability report, except Imperial Oil (2022, latest available). Bill C-59 stripped environmental content from the CNQ, Suncor and Cenovus 2024 reports, so their climate and disclosure scores are disclosure gaps, not performance verdicts. Credit models: 2024 annual financials from Yahoo Finance, USD reporters converted at USD/CAD 1.435. Merton is point-in-time at 31 December 2024. Transition data: Statistics Canada tables 25-10-0015-01, 25-10-0029-01 and 34-10-0035-01. Past performance does not predict future performance.